In [1]:
#setup
import sys
import pandas as pd
sys.path.append("../src")
from db_connection import get_engine

engine = get_engine()

In [2]:
#loading clean data from PostgreSQL
customers = pd.read_sql("SELECT * FROM clean_customers", engine)
transactions = pd.read_sql("SELECT * FROM clean_transactions", engine)
feedback = pd.read_sql("SELECT * FROM clean_feedback", engine)

**Customer Features (Churn Model)**

In [3]:
#basic features
features = customers.copy()

In [4]:
#creating tenure buckets
features['tenure_group'] = pd.cut(
    features['tenure'],
    bins=[0, 12, 24, 48, 100],
    labels=['0-1yr','1-2yr','2-4yr','4yr+']
)

In [5]:
#create revenue feature
features['total_revenue'] = features['monthlycharges'] * features['tenure']

In [6]:
#normalizing charges
features['avg_monthly_spent'] = features['totalcharges'] / (features['tenure'] + 1)

**Transaction features (Risk signals)**

In [7]:
#aggregating transaction behavior
txn_features = transactions.groupby('class').agg({
    'amount': ['mean','max','min','std']
}).reset_index()

In [8]:
#creating risk indicator
feedback['text_length'] = feedback['text'].apply(len)

In [9]:
#counting words
feedback['word_count'] = feedback['text'].apply(lambda x: len(x.split()))

In [10]:
#calculating average sentiment
avg_sentiment = feedback['sentiment'].mean()
print("Average sentiment:", avg_sentiment)

Average sentiment: 0.843980951439102


**Feature Selection**

In [ ]:
#selecting uselful features
selected_features = features[[
    'tenure',
    'monthlycharges',
    'totalcharges',
    'total_revenue',
    'avg_monthly_spent',
    'churn'
]]
#ID, text fields and high cardinality categories were dropped

**Handling categorical variables**

In [12]:
#converting categories
selected_features = pd.get_dummies(selected_features, drop_first=True)

**Train/Test Split**

In [14]:
from sklearn.model_selection import train_test_split

X = selected_features.drop('churn', axis=1)
y = selected_features['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Saving feature Table**

In [15]:
selected_features.to_sql("feature_customers", engine, if_exists="replace", index=False)

43

In [16]:
#checking distribution
X.describe()

,tenure,monthlycharges,totalcharges,total_revenue,avg_monthly_spent
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,32.371149,64.761692,2281.916928,2279.581350,61.173413
std,24.559481,30.090047,2265.270398,2264.729447,61.019723
min,0.000000,18.250000,18.800000,0.000000,9.183333
25%,9.000000,35.500000,402.225000,394.000000,26.274411
50%,29.000000,70.350000,1397.475000,1393.600000,61.150000
75%,55.000000,89.850000,3786.600000,3786.100000,84.940047
max,72.000000,118.750000,8684.800000,8550.000000,1397.475000
